# Factor Portfolio: Performance & Backtesting

Loads factor returns (VAL, MOM, LIQ, QLT, YLD), illustrates performance, and runs a simple backtest (equal-weight factor portfolio).

## 1. Setup and load factor returns

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Assume notebook is run from project root (Minerva Code/) where outputs/ exists
ROOT = Path.cwd()
if not (ROOT / 'outputs').exists():
    # Try parent (e.g. if opened from src/)
    ROOT = Path.cwd().parent
if not (ROOT / 'outputs').exists():
    raise FileNotFoundError('Run run_pipeline.py or run_all.py first to create outputs/')

# Load from portfolio_combined (output of run_all.py / run_pipeline.py)
factor_path = ROOT / 'outputs' / 'portfolio_combined' / 'factor_returns.xlsx'
if not factor_path.exists():
    factor_path = ROOT / 'outputs' / 'factors' / 'value_regional.xlsx'  # fallback: build from regional

def load_factor_returns_combined():
    """Load combined factor returns (VAL, MOM, LIQ, QLT, YLD)."""
    path = ROOT / 'outputs' / 'portfolio_combined' / 'factor_returns.xlsx'
    if path.exists():
        df = pd.read_excel(path, index_col=0)
        df.index = pd.to_datetime(df.index)
        return df
    # Build from regional files
    factors_dir = ROOT / 'outputs' / 'factors'
    out = {}
    for n, f in [
        ('VAL', 'value_regional.xlsx'), ('MOM', 'momentum_regional.xlsx'),
        ('LIQ', 'liquidity_regional.xlsx'), ('QLT', 'quality_regional.xlsx'),
        ('YLD', 'beta0_regional.xlsx'),
    ]:
        p = factors_dir / f
        if p.exists():
            df = pd.read_excel(p, index_col=0)
            df.index = pd.to_datetime(df.index)
            col = n if n in df.columns else ('BETA0' if n == 'YLD' and 'BETA0' in df.columns else None)
            if col:
                out[n] = df[col]
    return pd.DataFrame(out) if out else pd.DataFrame()

returns = load_factor_returns_combined()
returns = returns.dropna(how='all')
print('Factor returns shape:', returns.shape)
print(returns.head())

## 2. Performance metrics (annualized)

In [ ]:
def performance_stats(df):
    ann_ret = df.mean() * 12
    ann_vol = df.std() * np.sqrt(12)
    sharpe = ann_ret / ann_vol.replace(0, np.nan)
    cum = (1 + df).cumprod()
    cum_ret = cum.iloc[-1] - 1
    dd = (cum - cum.cummax()) / cum.cummax()
    max_dd = dd.min()
    return pd.DataFrame({
        'Ann. Return': ann_ret,
        'Ann. Vol': ann_vol,
        'Sharpe': sharpe,
        'Cum. Return': cum_ret,
        'Max Drawdown': max_dd
    })

stats = performance_stats(returns)
display(stats.round(4))

## 3. Performance charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Cumulative returns
ax = axes[0, 0]
cum = (1 + returns).cumprod()
cum.plot(ax=ax)
ax.set_title('Cumulative returns (1$ invested)')
ax.legend(loc='upper left', fontsize=8)
ax.set_ylabel('Cumulative return')
ax.grid(True, alpha=0.3)

# Sharpe by factor
ax = axes[0, 1]
stats['Sharpe'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='navy')
ax.set_title('Annualized Sharpe ratio by factor')
ax.set_ylabel('Sharpe')
ax.axhline(0, color='gray', linestyle='-')

# Drawdowns over time
ax = axes[1, 0]
cum = (1 + returns).cumprod()
dd = (cum - cum.cummax()) / cum.cummax()
dd.plot(ax=ax)
ax.set_title('Drawdown by factor')
ax.set_ylabel('Drawdown')
ax.legend(loc='lower left', fontsize=8)
ax.grid(True, alpha=0.3)

# Rolling 12-month Sharpe (example: first column or equal-weight)
ax = axes[1, 1]
ew = returns.mean(axis=1)
roll_ret = ew.rolling(12).mean() * 12
roll_vol = ew.rolling(12).std() * np.sqrt(12)
roll_sharpe = (roll_ret / roll_vol.replace(0, np.nan)).dropna()
roll_sharpe.plot(ax=ax)
ax.set_title('Rolling 12m Sharpe (equal-weight factor portfolio)')
ax.set_ylabel('Sharpe')
ax.axhline(0, color='gray', linestyle='-')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Backtest: equal-weight factor portfolio

In [ ]:
# Strategy: each month invest equally in VAL, MOM, LVOL, QLT
weights = 1.0 / returns.shape[1]
strategy_returns = returns.mean(axis=1)
strategy_returns.name = 'EqualWeight'

strat_stats = performance_stats(pd.DataFrame({'EqualWeight': strategy_returns}))
print('Equal-weight factor portfolio:')
display(strat_stats.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
cum_strat = (1 + strategy_returns).cumprod()
cum_strat.plot(ax=ax, label='Equal-weight factor portfolio')
ax.set_title('Backtest: cumulative return')
ax.set_ylabel('Cumulative return')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Backtest: rolling-window mean–variance (optional)

In [ ]:
# Use past 24 months to estimate mean and covariance; maximize Sharpe (or min vol)
lookback = 24
strat_rets = []
for t in range(lookback, len(returns)):
    hist = returns.iloc[t - lookback:t]
    mu = hist.mean()
    cov = hist.cov()
    # Minimum volatility portfolio (or max Sharpe with constraint)
    inv_cov = np.linalg.pinv(cov)
    ones = np.ones(len(mu))
    w = inv_cov @ ones / (ones @ inv_cov @ ones)
    w = pd.Series(w, index=returns.columns)
    r = (w * returns.iloc[t]).sum()
    strat_rets.append(r)

idx = returns.index[lookback:]
strat_series = pd.Series(strat_rets, index=idx)
strat_series.name = 'MinVol'
mv_stats = performance_stats(pd.DataFrame({'MinVol': strat_series}))
print('Rolling min-vol factor portfolio (24m window):')
display(mv_stats.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
(1 + strategy_returns).cumprod().plot(ax=ax, label='Equal-weight')
(1 + strat_series).cumprod().plot(ax=ax, label='Rolling min-vol (24m)')
ax.set_title('Backtest comparison')
ax.set_ylabel('Cumulative return')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()